# Sample only - license and disclaimer

Copyright (c) 2026 The notebook contributors. Licensed under the [MIT License](./LICENSE). Retain the license and copyright notice when copying or distributing this notebook.

**This notebook is a sample for demonstration and educational purposes only. It is not production-ready or an officially supported Microsoft tool. You must review, validate, and test it in a non-production environment before running it in production.**

This notebook creates or changes OneLake data access roles and can change who can access your data. Review the target workspace, lakehouse, tables, groups, existing role permissions, and printed API payloads before use.

**You run this notebook at your own risk. You are responsible for deciding whether it is suitable for your environment, obtaining authorization, testing, backups, recovery, security, and compliance. You assume responsibility for your use, modifications, and deployment, and for any resulting issues or consequences, including unintended access, data exposure or loss, service disruption, and costs.**

The notebook is provided **AS IS**, without warranties or guarantees. To the fullest extent permitted by applicable law, the authors and copyright holders disclaim liability for claims, damages, or other liability arising from its use, as set out in the MIT License. Nothing in this notice excludes liability that cannot lawfully be excluded.

This notice explains usage risks and does not add restrictions to the permissions granted by the MIT License.

In [ ]:
workspace_id = "<replace-with-your-workspace-id>"
lakehouse_id = "<replace-with-your-lakehouse-id>"

# Name of OneLake sec role - this noteobook processes one role at a time
name_of_role = "<replace-with-OneLake-sec-role-name>"

# Name of tables. This can be one/more tables separated by comma.
# tables = "<schema1>.<table1>, <schema2>.<table2>"
tables = "<replace-with-table-name" # This can be a list

# Replace with your security group object IDs.
# This must be a list of valid security group object IDs.
# Example: security_group_ids = ["sec object id 1","sec object id 2",...,"sec object id N"]
security_group_ids = ["<replace-with-security-group-object-id>"]

In [ ]:
import base64
import json
from urllib.parse import quote
import requests



# Fetch the token required for interacting with REST API
token = notebookutils.credentials.getToken("https://api.fabric.microsoft.com")
token_part = token.split(".")[1]
tenant_id = json.loads(base64.urlsafe_b64decode(token_part + "=" * (-len(token_part) % 4)))["tid"]
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Construct the payload for creating a new data access role
new_role = {
    "name": name_of_role,
    "kind": "Policy",
    "decisionRules": [{
        "effect": "Permit",
        "permission": [
            {"attributeName": "Path", "attributeValueIncludedIn": [
                "Tables/" + "/".join(part.strip() for part in table.strip().split("."))
                for table in tables.split(",")
            ]},
            {"attributeName": "Action", "attributeValueIncludedIn": ["Read"]},
        ],
    }],
    "members": {"microsoftEntraMembers": [
        {"objectId": group_id, "tenantId": tenant_id} for group_id in security_group_ids
    ]},
}

# Print the payload for verification before making the API call
print("Create payload (Abort):")
print(json.dumps(new_role, indent=2))

# Make the API call with conflict polict set to abort. This ensures that admin has visibility and knows that role already exists before attempting to overwrite it.
create_response = requests.post(
    f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{lakehouse_id}/dataAccessRoles?preview=true&dataAccessRoleConflictPolicy=Abort",
    headers=headers,
    json=new_role,
    timeout=60,
)

# If-then-else block
# If API call is successful, print the response OK.
# If it fails, proceed to next step which is to fetch the existing role, merge the new permissions and members, and then overwrite the role.
if create_response.ok:
    print("Create:", create_response.status_code, create_response.text)
else:
    print("Create failed:", create_response.status_code, create_response.text)
    get_response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{lakehouse_id}/dataAccessRoles/{quote(name_of_role, safe='')}?preview=true",
        headers=headers,
        timeout=60,
    )
    print("GET:", get_response.status_code, get_response.text)
    role_payload = get_response.json()
    existing_permissions = role_payload["decisionRules"][0]["permission"]
    for new_permission in new_role["decisionRules"][0]["permission"]:
        for permission in existing_permissions:
            if permission["attributeName"] == new_permission["attributeName"]:
                for value in new_permission["attributeValueIncludedIn"]:
                    if value not in permission["attributeValueIncludedIn"]:
                        permission["attributeValueIncludedIn"].append(value)
                break
        else:
            existing_permissions.append(new_permission)
    existing_members = role_payload.setdefault("members", {}).setdefault("microsoftEntraMembers", [])
    for new_member in new_role["members"]["microsoftEntraMembers"]:
        if not any(
            member["tenantId"].lower() == new_member["tenantId"].lower()
            and member["objectId"].lower() == new_member["objectId"].lower()
            for member in existing_members
        ):
            existing_members.append(new_member)
    print("Overwrite payload:")
    print(json.dumps(role_payload, indent=2))

    overwrite_response = requests.post(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{lakehouse_id}/dataAccessRoles?preview=true&dataAccessRoleConflictPolicy=Overwrite",
        headers=headers,
        json=role_payload,
        timeout=60,
    )
    print("Overwrite:", overwrite_response.status_code, overwrite_response.text)

In [ ]:
# Doing post checks 
# Call GET API to fetch the existing role details

get_response = requests.get(
        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items/{lakehouse_id}/dataAccessRoles/{name_of_role}?preview=true",
        headers=headers)

print("Role details:", get_response.status_code, get_response.text)